*Most of the strong agents in this competition are not programs that play the game. They are a recorded season with a thin layer of code around it.*

If you open a high scoring public agent expecting to find a strategy, you find a compressed blob and about three hundred lines that mostly repair things. That is not laziness and it is not a trick, it is a close to forced response to three properties of this engine.

This notebook is a field guide to that category. What one of these agents is made of, the eight kinds of layer you can wrap around a recording and which of them anybody actually ships, how to classify an agent whose source you will never see, and where the ceiling of the whole approach sits.

**The layer is worth much less than it looks and the recording is worth much more.** Two agents carrying the same layer finish sixty thousand apart, and what separates them was decided before a line of the layer ran.

Every measurement in this guide that involves playing games ran on an engine-exact C++ port of the environment, open at [github.com/destbreso/kaggriculture-cppsim](https://github.com/destbreso/kaggriculture-cppsim) and validated in [its own notebook](https://www.kaggle.com/code/destbreso/kagsim-the-engine-at-2000-episodes-per-second) by reproducing recorded ladder banks to the dollar; the pure-measurement results come from public replays anyone can pull.


In [ ]:
import math
import matplotlib.pyplot as plt

BLUE, ORANGE = "#2F6690", "#C8663A"
RAMP = ["#A8C3D9", "#5C8CB4", "#2F6690"]        # magnitude, light to dark
INK, MUTED, GRID = "#1F2933", "#6B7280", "#E3E7EB"

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 9.5,
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED,
    "axes.titlesize": 10.5, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlepad": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.axisbelow": True,                     # grid under the marks, not through

    "xtick.color": MUTED, "ytick.color": MUTED,
    "grid.color": GRID, "grid.linewidth": 0.8, "figure.facecolor": "white",
})

## 1. What one of these agents is

A tape and a rim.

The tape is a list of 719 or 720 action dictionaries, one per turn, each with the three keys the engine expects: what the farmer does, what each hired hand does, and what orders go to the market. It is a season somebody played, stored verbatim.

The rim is the code around it, and at minimum it is this.

In [ ]:
def agent(obs):
    turn = obs["day"] * 24 + obs["hour"]         # never obs["step"], see below
    if turn < len(TAPE):
        return TAPE[turn]
    return {"farmer": ["PASS"], "hands": [], "market": []}

That is a complete, functional, competitive agent. Everything else any of them ships sits on top of it.

**About that clock comment, because the story closed since this was first drafted.** `obs["step"]` turned out to be a serialization asymmetry, not a live handicap: the live `env.run()` dispatch injects the shared step for both seats right before your callable runs, but the RECORDED observations, `env.steps` and every downloaded `replay.json`, carry `step: None` for seat 1 on all 719 turns. So an agent that reads `step` plays fine on the ladder and breaks in every replay-driven harness, backtest, or analysis pipeline, including yours. The derive-from-day-and-hour form above is exact in all three worlds, which is why it stays the rule even now that the live side is exonerated. Two public agents published the same week are a natural A/B: fed serialized seat-1 observations, the one carrying the fallback banks 48,850 where the one reading step-or-zero banks its untouched starting 3,000.

The word "agent" suggests something the accounting does not support.


Measured by decompacting all sixteen: for each agent, the payload bytes that decode to recorded ACTIONS rather than to source. The distribution is the finding, so the agents are shown unnamed; the point is the shape of the category, not a league table.


In [ ]:
# Sixteen public agents, decompacted. Tape share = fraction of the payload
# that decodes to recorded actions; code lines = the visible source around it.
shares = [99, 94, 75, 75, 75, 72, 66, 65, 56, 55, 54, 42, 40, 31, 0, 0]
lines  = [26, 196, 908, 911, 911, 1059, 1076, 742, 209, 212, 219, 354, 381, 636, 2142, 246]

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.4))
y = range(len(shares))
ax = axes[0]
ax.barh(list(y), shares, color=[BLUE if s else MUTED for s in shares], height=0.65)
ax.set_yticks(list(y), [f'agent {chr(65+k)}' for k in y], fontsize=8)
ax.invert_yaxis(); ax.set_xlabel('tape share of payload, %')
ax.set_title('most of the agent is a recording', loc='left')
ax = axes[1]
ax.scatter(lines, shares, s=42, color=BLUE)
ax.set_xlabel('lines of visible code'); ax.set_ylabel('tape share, %')
ax.set_title('and code volume does not buy tape freedom', loc='left')
for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()


Two of those sixteen are not what they appear. One reports no tape because its recordings live inside source modules it unpacks at runtime, so the byte counter never reaches them: read that bar as unmeasured, not as zero. And the counter finds module-level payloads only, which is why another agent briefly looked like seven thousand lines of code when it is twenty-six lines wrapped around its own season.

The one agent with genuinely no tape is the exception that makes the point. It is the only agent in the set written the way a beginner expects an agent to be written, rules over the state, 271 competent lines, and it scores at the bottom of the group. The category exists because it wins.


**Why a recording is legal in the first place.** Three properties of this engine, and a route needs all three.

**Determinism.** Given the seed and both action streams, the episode is a pure function. Randomness enters at exactly two places, both at the end of a day: weeds spawning on empty tiles, and a town shop unlocking every three days.

**Silent rejection.** An action that is not legal in the current state is a no-op. No exception, no signal, no log line. A tape that tries to harvest a tile that is not ready simply does nothing that turn, so a route recorded in one game does not crash in another.

**Complete observation.** Each agent sees both farms in full, every tile, every animal, the whole market book. Only the opponent's shed is hidden. Nothing about replaying a plan requires guessing at your own state.

The second property is what makes replay robust, and it is also the trap: the same silence hides a broken agent. A market list longer than ten orders is truncated without a word, and a shed at capacity refuses purchases the same way.


## 2. The one piece of engine you have to hold in your head

Everything a layer can do goes through the market, so the market's price function is the whole mechanism. It is short enough to read.

In [ ]:
# transcribed from the engine at 1.32.7, verified exact in 1,863 of 1,863
# quotes against the real market_price
PRICE_FLOOR, I0 = 1, 10_000
P = {  # base, knee T, curve below the neutral inventory, curve above it
    "WHEAT":      (25,  400, "sqrt",  0.8, "log",    0.2),
    "CARROT":     (35,  450, "hinge", 1.0, "sqrt",   0.7),
    "TOMATO":     (60,  200, "hinge", 0.4, "sqrt",   0.6),
    "STRAWBERRY": (120, 100, "sqrt",  0.7, "linear", 1.6),
    "MELON":      (250, 300, "log",   0.2, "sq",     3.6),
    "EGG":        (50,  332, "hinge", 0.4, "log",    0.2),
    "MILK":       (160, 122, "sqrt",  0.6, "linear", 1.6),
    "WOOL":       (200, 105, "log",   0.2, "sq",     3.2),
}

def shape(f, x, T):
    x = max(0.0, x)
    if f == "linear": return x
    if f == "sq":     return x * x
    if f == "sqrt":   return math.sqrt(x)
    if f == "hinge":                                    # new in 1.32.7
        u = x / T
        return u if u <= 1.0 else u + 8.0 * (u - 1.0) ** 2
    return math.log(1.0 + x)

def price(item, inventory):
    base, T, bf, bt, af, at = P[item]
    if inventory < I0:
        amp = bt * base / shape(bf, T, T)
        v = base + amp * shape(bf, I0 - inventory, T)
    else:
        amp = at * base / shape(af, T, T)
        v = base - amp * shape(af, inventory - I0, T)
    return max(PRICE_FLOOR, int(round(v)))              # <-- an INTEGER

print("selling milk one unit at a time into a neutral book:")
for k in range(6):
    print(f"  unit {k + 1}: ${price('MILK', I0 + k)}")

Two things in that function decide most of what follows.

**The price is an integer and the last line rounds.** It is not a smooth function of quantity, it is a staircase, and the steps are far wider than they look: selling wheat into a neutral book, the median step is 46 units at exactly the same quote and the widest is 154. Then one more unit costs you a dollar on every unit after it. Anything that reasons about this market by taking a derivative is reasoning about a curve the engine does not have, which is why the optimal-control literature has less to offer here than it first appears.

**The scarcity branch has a knee.** Three products, carrot, tomato and egg, got a convex branch in the August 2026 balance change: linear up to the knee and quadratic past it. Below the knee it behaves, above it the price runs away.

In [ ]:
def flat_runs(item, span=300):
    """How many units sell at the SAME quoted price, step by step."""
    runs, cur, prev = [], 1, price(item, I0)
    for k in range(1, span):
        q = price(item, I0 + k)
        if q == prev:
            cur += 1
        else:
            runs.append(cur); cur, prev = 1, q
    runs.append(cur)
    return sorted(runs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 3.6))

# LEFT: the staircase, the thing a derivative cannot see
sold = list(range(0, 260))
ax1.step(sold, [price("WHEAT", I0 + s) for s in sold], where="post",
         color=BLUE, lw=2)
ax1.set_title("The price is a staircase, not a curve")
ax1.set_xlabel("units of wheat sold into a neutral book")
ax1.set_ylabel(r"\$ per unit")
ax1.grid(axis="y")
r = flat_runs("WHEAT")
med = r[len(r) // 2]
ax1.annotate(f"the median step is {med} units wide.\n"
             f"Sell {med} and the quote does not move.\n"
             f"Sell one more and it costs a dollar\non every unit after it.",
             xy=(140, price("WHEAT", I0 + 140)),
             xytext=(0.30, 0.62), textcoords="axes fraction",
             color=INK, fontsize=8.5, va="top",
             arrowprops=dict(arrowstyle="->", color=MUTED, lw=1))
ax1.set_ylim(price("WHEAT", I0 + 259) - 0.6, price("WHEAT", I0) + 0.6)

# RIGHT: the 1.32.7 hinge, against a same-shaped control
short = list(range(0, 600))
ax2.plot(short, [price("TOMATO", I0 - s) for s in short], color=ORANGE, lw=2)
ax2.plot(short, [price("STRAWBERRY", I0 - s) for s in short], color=BLUE, lw=2)
ax2.axvline(P["TOMATO"][1], color=MUTED, lw=1, ls=(0, (4, 3)))
ax2.set_title("And past the knee it runs away")
ax2.set_xlabel("units of scarcity below the neutral book")
ax2.set_ylabel(r"\$ per unit")
ax2.set_xlim(0, 760)
ax2.grid(axis="y")
ax2.text(612, price("TOMATO", I0 - 600), " tomato, hinge", color=ORANGE,
         fontsize=9, va="center", fontweight="bold")
ax2.text(612, price("STRAWBERRY", I0 - 600), " strawberry, sqrt", color=BLUE,
         fontsize=9, va="center", fontweight="bold")
ax2.text(216, ax2.get_ylim()[1] * 0.9, "knee, T = 200", color=MUTED, fontsize=8)
fig.tight_layout()
plt.show()

The right hand panel is the whole reason the balance change mattered, and it is also a warning about how to read it.

Games do reach that branch. Replaying 120 real episodes with nothing of mine in the loop, the book passes the tomato knee in 55 % of them, carrot in 28 % and egg in 26 %, which reproduces the rates staff announced by a method sharing nothing with theirs. Melon, as a control, passes its knee in zero of 120.

But the ledgers did not move, and the reason is that **the field never sells these three**. The branch is hot and the ledgers are cold. What is worth nothing in the median game is worth about a doubling at the ninetieth percentile, which makes it a dispersion instrument rather than a revenue one, and dispersion has a sign here: when you are behind, more of it is what you want.

**The market list is resolved by index.** The market list is resolved **by index**, one order at a time, against the money and the shed as they stand at that moment.

So `[SELL wool, BUY_LAND]` buys the quadrant on five dollars of starting cash, and `[BUY_LAND, SELL wool]` buys nothing at all. Same turn, same two orders, different game.

That is a financing chain, and every recorded tape is full of them: a sale sitting at a lower index than the purchase it pays for, on the same turn. It also means the recorded sells are not only a trading schedule, they are a **financing schedule**, and deleting one silently turns a land purchase into a no-op that leaves half the farm locked for the rest of the season.

The cap is ten orders per turn and the eleventh is dropped without a word.


## 3. The field that makes a recording work

A recording is a plan timed against the state it expects, and the state it expects includes the opponent. Replay works here because the opponent is, most of the time, running a plan too.

Three measurements of that, each by a different method.

**94 % of seats share a schedule at turn 24, and 23 % still do at turn 400.** Counted by byte-exact hash identity over 436 seats. What survives to turn 400 is 36 distinct schedules covering 100 seats, and the largest of them holds 16 seats across 9 teams and never breaks step at all, identical from turn 1 to turn 718. Its best ranked member was #1.

**Three unrelated top teams play 67 to 80 % of one public notebook route.** Measured against the route decoded from a published notebook: 80.2, 69.7 and 66.8 percent, one of them holding rank 4 at the time. All three act on 100 % of turns, so none of this is agreement about doing nothing.

**And the agreement is front-loaded**, which locates where any edge has to come from.

In [ ]:
# docs/FIELD_DRIFT.md section 7: pairwise turn-by-turn agreement between two
# top-ten routes, by in-game day band. Reproduced independently by the schedule
# census, which counts byte-exact hash identity instead of pairwise agreement.
BANDS = ["0-3", "3-7", "7-14", "14-21", "21-30"]
A = [93.1, 77.1, 75.6, 61.9, 56.7]          # top-ten route pair 1
B = [90.3, 79.2, 59.5, 45.8, 40.5]          # top-ten route pair 2

fig, ax = plt.subplots(figsize=(7.4, 3.2))
x = range(len(BANDS))
ax.plot(x, A, color=BLUE, lw=2, marker="o", ms=6)
ax.plot(x, B, color=ORANGE, lw=2, marker="o", ms=6)
ax.set_xticks(list(x), BANDS)
ax.set_ylim(0, 100)
ax.set_xlabel("in-game days")
ax.set_ylabel("turn-by-turn agreement, %")
ax.set_title("Two top-ten routes agree at the start and part at the end")
ax.grid(axis="y")
ax.text(4.08, A[-1], f" {A[-1]:.0f} %  rank 4 vs rank 19", color=BLUE,
        va="center", fontsize=8.5, fontweight="bold")
ax.text(4.08, B[-1], f" {B[-1]:.0f} %  rank 4 vs route pair", color=ORANGE,
        va="center", fontsize=8.5, fontweight="bold")
ax.set_xlim(-0.2, 6.6)
# the margin is generated LATE, so the shading goes on the day axis
ax.axvspan(2.5, 4.0, color=GRID, alpha=0.55, zorder=0)
ax.text(3.25, 6, "where the margin\nis generated", color=MUTED, fontsize=8.5,
        ha="center")
fig.tight_layout()
plt.show()

The two routes agree on 90 to 93 % of days 0 to 3 and on 41 to 57 % of days 21 to 30. So a head-to-head margin between two agents in this field is generated in the tail, in the fifth of the stream where they stop being the same agent.

**That is a fragility, not a comfort.** If most of what you field is public and widely replayed, your edge lives in the 20 to 30 % that differs, and it is thin.

Two more properties of the field, both of which change how you should measure.

**The field turns over by successive sweeps**, with a measured half life of 2.1 days. A clade takes a quarter to a half of all seats in two days, holds three, and is extinct in five. One clade holds 41 of the 69 submissions ever rated above 2,900. Any instrument calibrated against "the field" expires on that schedule.

**And a pool of forty opponents is not forty.** Cluster 40 recorded opponents at 90 % turn-by-turn agreement and they collapse to **19 behaviours**, with one cluster holding 18 of them. Sampling uniformly from a clustered population gives you an effective size governed by the number of clusters, not the number of draws.

**The distinction that decides whether a recording can represent an opponent.** Divergence alone cannot tell an adaptive policy from a second fixed script. Both diverge.

The test that can: take a team with two recorded episodes at **different seeds against different opponents**, find where the streams split, then measure how often they agree after the split. A live policy meeting a different board and a different rival does not re-converge. A script with local repairs does.

**The #1 agent is a script with repairs.** The genuinely adaptive opposition is three teams inside the top 25, and they are the only opponents in this competition that a recording cannot represent.


## 4. What you may change in a tape, and what you may not

This is the question that decides whether a layer is an improvement or a different agent, and it has an answer that costs no games.

Take one team's recorded episodes. Align them against each other and every turn falls into one of three states.

**Frozen**: the plan aligned and the market order matched. The source never varied here.

**Mutable**: the plan aligned and the market order differed. **The source itself varied here with its plan intact**, so a change here is supported by evidence rather than guessed.

**Off limits**: the plan did not align at all.

The baseline is the **medoid**, the recorded episode whose alignment against the others is highest. A real episode, so it is a legal action sequence by construction, which a position-by-position consensus of two legal routes need not be. And it self-diagnoses: an agent with one plan scores near 1, an agent with none cannot, so the method reports its own inapplicability instead of returning a confident answer.

Measured over 1,492 real episodes of the top ten.

In [ ]:
# docs/GENETIC_POTENTIAL.md, 1,492 real episodes of the top ten.
# Points are labelled by RANK: two of the ten teams carry names in scripts
# matplotlib has no glyphs for, and a chart must not print a box where a name
# goes. The names are in the table below.
TEAMS = [   # rank, plan aligned %, mutable turns, medoid
    (2, 83, 142, 0.989), (8, 81, 140, 0.993), (9, 81, 35, 0.984),
    (5, 32, 19, 0.816),  (3, 25, 21, 0.914),  (4, 24, 29, 0.702),
    (1, 19, 23, 0.474),
]
OFF = {2: (0, 1.1), 8: (0, -1.4), 9: (0, 1.1), 5: (0, -1.4),
       3: (0, 1.1), 4: (0, 1.1), 1: (0, -1.4)}

fig, ax = plt.subplots(figsize=(7.8, 4.0))
for rank, plan, mut, medoid in TEAMS:
    usable = medoid >= 0.9
    y = 100 * mut / 719
    ax.scatter(plan, y, s=190 if usable else 150,
               color=BLUE if usable else ORANGE,
               marker="o" if usable else "X", zorder=3,
               edgecolor="white", linewidth=1.3)
    dx, dy = OFF[rank]
    ax.annotate(f"rank {rank}", (plan + dx, y + dy), fontsize=9,
                ha="center", color=INK, fontweight="bold", zorder=4)
ax.set_xlabel("share of turns whose PLAN repeats across the team's own episodes, %")
ax.set_ylabel("turns the source itself varied, %")
ax.set_title("Holding a plan and varying a market are two different things")
ax.grid(True)
ax.set_xlim(10, 95)
ax.set_ylim(-2.5, 24)
ax.scatter([], [], color=BLUE, s=120,
           label="medoid 0.9 or better: a fixed plan exists to take")
ax.scatter([], [], color=ORANGE, s=110, marker="X",
           label="medoid below 0.9: the method reports itself inapplicable")
ax.legend(frameon=False, fontsize=8.5, loc="upper left")
fig.tight_layout()
plt.show()

| rank held then | | plan aligned | mutable turns | of its trading turns | medoid |
|---:|---|---:|---:|---:|---:|
| 2 |  | **83 %** | **142 (19.7 %)** | 42.6 % | 0.989 |
| 8 |  | **81 %** | 140 (19.4 %) | 45.5 % | 0.993 |
| 9 |  | **81 %** | 35 (4.9 %) | **12.5 %** | 0.984 |
| 5 |  | 32 % | 19 (2.6 %) | 29.2 % | 0.816 |
| 3 |  | 25 % | 21 (2.9 %) | 45.7 % | 0.914 |
| 4 |  | 24 % | 29 (4.0 %) | 65.9 % | 0.702 |
| 1 |  | 19 % | 23 (3.2 %) | 65.7 % | **0.474** |

Two things fall out, and both are counterintuitive.

**Plan-fixity and market-reactivity are separate axes.** Rank 9 holds its plan more tightly than almost anyone and varies its market on only 12.5 % of the turns it trades. Ranks 2 and 8 hold their plans just as tightly and vary on 43 to 46 %. So "fixed plan plus reactive seller" is not one property, it is two, and three of the four cells are occupied.

**The best agent on the board is the least usable donor.** Rank 1 is the strongest agent in the competition and at a medoid of 0.474 it barely repeats itself, so there is no fixed plan there to take. That number fires before any conclusion is drawn, which is the whole point of carrying a self-diagnosing baseline.

Two structural findings settle what an edit may look like at all.

**Construction is never reordered.** Of the plan actions that fail to align, the share that could be explained by a block having moved is 0.46 % over all ten agents.

**And blocks are barely even displaced.** 288,378 of 288,648 matched positions, **99.91 %**, sit at the same index. Four agents show zero displacement at all. When an offset does occur it can be large, up to 144 turns, so displacement is rare rather than small.

**Mutable does not mean rewritable, and the failure is total rather than proportional.** The measurement above treats `market` as one channel, meaning the whole list. **The engine accepts six operations in that list and three of them change the FARM**: `HIRE`, `BUY_LAND` and `BUY_ANIMAL`.

Hands are dismissed every night and re-hired every morning, so a hand that was never hired cannot execute the `hands` action recorded for it. A quadrant never bought cannot be planted.

So a mutable turn means the source varied here. It does **not** mean this may be rewritten. For the two agents that are genuine market mutants the difference looks small: of 131 and 95 mutable turns, 129 and 95 involve `SELL` and none involve inputs.

It is not zero, and what it hides does not degrade gracefully. **One of my own agents deleted the sell channel, a `BUY_LAND` went unfunded, and 50 tiles stayed locked against the opponent's 25 for the rest of the season.**

The reason is the index rule from section 2: the recorded sells sit at a lower index than the purchase they pay for, in the same list, on the same turn. **Tolerance of substitution is not evidence of benefit, and it is not evidence of safety either.**


### What kills a tape away from home, located to the coin

A recording is a plan AND a financing schedule: its purchases assume the revenues of the game where it was recorded. Move it to a different seed, or put a rival beside it who sells into the same book, and the early sells realise a few dollars less, a purchase comes up short by cents, and the engine refuses it IN SILENCE.

One forensic run of that mechanism, day by day: the same tape banks 150,914 alone on a world and 43,929 against a co-seller on the same world. The whole difference is days 4 to 9, cash pinned at zero, four or five purchases silently refused, and by day 10 the farm has a QUARTER of the strawberry tiles and HALF the herd it has in the solo run, with 32 unsold wheat sitting in the shed while the purchases failed. Nothing dramatic ever happens; the game is lost in refusals of forty dollars each.

**And the mirror lesson for anyone editing tapes.** Repairs of this exist, and they cut both ways: the same inserted sell that rescues the establishment in a cascade world sells the herd's feed buffer in an ordinary one, where two unfed days kill an animal and 200 of wheat costs 25,000 of cows. Every discriminator between those two regimes that reads a single turn has failed here so far.

### The seed-variance law

The mutable surface says where the source itself varied, and it is still not a licence to tune. A table of per-world tape edits confirmed at TWO seeds per world went 3 wins, 11 losses, 12 ties at a THIRD seed; four fresh specialist tapes that each beat the incumbent at one seed of their world all lost at a sibling seed, one by 44,000. The pattern held four separate times in one week: a perturbation of a tape validated on N seeds, selected on n much smaller than N, keeps its screen gain only by luck. The edits that survive are the ones that never needed the selection to be right, monotone by construction or carrying a proven loss floor.


In [ ]:
# The selection funnel that ate itself: per-world edits of a tape, measured
# at one seed, a second, and then a third the selection never touched.
stages = ['gain at the\nscreen seed', 'cleared the\nthreshold', 'survived a\nsecond seed', 'still winning\nat a third']
n      = [46, 37, 13, 3]

fig, ax = plt.subplots(figsize=(7.6, 3.4))
ax.bar(range(4), n, color=[BLUE, BLUE, BLUE, ORANGE], width=0.62)
for k, v in enumerate(n):
    ax.text(k, v + 0.8, str(v), ha='center', fontsize=11, color=INK)
ax.set_xticks(range(4), stages, fontsize=9)
ax.set_ylabel('worlds (of 64)')
ax.set_title('two seeds of selection still overfit', loc='left')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()


## 5. Eight cells, and only five of them are occupied

"The adaptive layer" is not one thing. Split it two ways and the field sorts itself out.

A layer is defined by two independent choices.

**What it reads**: nothing, its own state, the world, or the opponent. That is the trigger.

**What it writes**: either an edit that conserves lifetime supply, moving *when* a unit is sold without changing *what* or *how much*, or an edit that does not conserve it.

Four triggers by two edit kinds is eight cells. Here is which of them anybody ships, counted over the eight public agents I read line by line, each reading checked by a second pass told to refute it.

| | reads | writes | name | ships | what it does |
|---|---|---|---|---|---|
| 0 | nothing | nothing | pure replay | 16/16 | the floor, and already competitive |
| 1 | own state | conserving | repair | 16/16 | retry what the tape asked for and could not have |
| 2 | own state | rewriting | projected shed | 4/8 | simulate this turn's deposits, then price the sale |
| 3 | world | conserving | draw-timed sells | **0/8** | nobody, and it is the empty cell worth trying |
| 4 | world | rewriting | route branch | 5/8 | carry five tapes, pick on which shop opened first |
| 5 | opponent | conserving | preempt ledger | 4/8 | sell earlier, record the advance, cancel it later |
| 6 | opponent | rewriting | counter-supply | **dark** | built, complete, switched off in three agents |
| 7 | any | composition | rebuild the farm | 0/16 | structurally out of reach, and see section 8 |

Three things follow from the occupancy column.

**Cell 1 is universal and nobody calls it adaptation.** Every one of these agents retries actions that failed, and that is the bulk of what their three hundred lines do. It is unglamorous and it is most of the value the rim adds.

**Cell 3 is empty.** Nobody times their selling off the shop draw, even though five of eight branch their whole route on the same signal one cell over. The shop draw is the largest exogenous swing in the game, it is observable from the turn it happens, and it needs no opponent model at all. That is the cheapest unclaimed thing on this list.

**Cells 5 and 6 are where the rest of this notebook goes.**

### What a thin layer can do, and what it cannot

The eight cells say what people ship. This is the harder question underneath: which KINDS of edit does a recorded season tolerate at all. Classes, not recipes; the intuition rides inline with each, because the intuition is the transferable part.

**What it can do.**

*Terminal liquidation.* At the last turns, append sells for whatever the shed still holds. The engine fills only what exists, so the edit is monotone by construction: its worst case is the tape unchanged. Anything with that property never needs a selection procedure to be right, which after the seed-variance section below you will recognise as the highest compliment an edit can receive.

*Capacity and geometry repair.* Pad or trim the hands commands to the crew that actually exists; when a planned PLANT lands on a weed the recorded game never grew, dig first and replay the intent a turn late. The tape asked for something the current board cannot grant; the repair grants the nearest legal thing. The intuition: the plan is right, the world moved, restore the mapping.

*Financing repair.* The tape's purchases assume the revenues of the game where it was recorded, and against a co-selling rival a purchase can come up short by a few coins and be refused in silence. A ledger that walks the emitted list, notices the shortfall, and inserts a sale of surplus stock BEFORE the purchase is repairing the plan's own intent. The intuition: the tape is not a list of actions, it is a cash-flow schedule, and you are allowed to defend the schedule. The sharp edge: surplus must mean surplus, because the same wheat that funds a seed batch is somebody's feed, and an animal starves in two missed days.

*Conserved substitution.* Keep the geometry, the count and the slots; change only WHICH animal a planned purchase buys, or WHICH product a planned sale sells, using demand that is visible on the board. The recorded plan carries the logistics; the substitution re-aims them. The intuition: the expensive thing in the plan is the choreography, not the label.

*Route selection at branch points.* Carry several complete recordings and pick between them on public state the seed fixed, the shop reveals above all. This is the strongest class in the field and it is barely a layer at all: every candidate action still comes from a coherent recorded season.

*Order-list hygiene.* Within one turn's market list, keep sells ahead of the purchases they finance, and cap sale quantities to what the shed will actually contain when the list settles. The engine resolves the list in order and refuses in silence; hygiene just stops the silence from costing money.

**What it cannot do.**

*Re-time sales against the price.* Moving a sale earlier or later to catch a better book sounds like the obvious edit and it is the one that keeps failing: pulled sales fight the tape's own financing downstream, and pushed sales walk into the price moves that motivated them. Without a full expected-value model of the book, timing edits are noise with a cost. The hindsight experiment in section 9 puts a ceiling on the whole channel, and it is small.

*Replan geometry.* Walks, build placement and the work calendar carry timing dependencies the recording never states. An edit that moves a unit somewhere else this hour has opinions about every later hour, and the tape will not negotiate.

*Tune per-world edits from a handful of games.* Any edit selected on fewer seeds than the tape's own validation keeps its measured gain only by luck. This is the seed-variance law below, and it eliminated four separate promising edits in one week here.

*Replace a channel wholesale.* The moment your layer's output is the default rather than the tape's, the burden inverts: you must now beat the recording everywhere at once, and you have become a policy author holding a tape as a security blanket. Every wholesale replacement measured here lost to the bare tape it replaced.


### The five mechanisms that are actually shipped

Each one read line by line in eight public agents, each reading checked by a second pass told to refute it. Ordered by how many independent authors ship it, because convergence across unrelated authors is the strongest available evidence that a technique pays.

#### Six of eight: price-impact ordering inside the turn

The market list settles by index, so the same orders in a different order are a different outcome. Every donor with a market layer ranks its sells by modelled price impact before emitting them. One calls it sell-slot reordering, two call it impact ranking, one calls it order-index front-running.

**Five of eight also transcribe the engine's price curve into their own file** so they can price a sale before making it. That is the function in section 2.

#### Four of eight: the projected shed, a one-turn lookahead

Before deciding what to sell, simulate this turn's `DROP` and `PLACE` so the market sees the shed the market will actually see, not the shed visible when the decision is taken. Without it every quantity is computed against an inventory one turn stale, and section 2's staircase means a stale inventory prices the sale on the wrong step.

#### Five of eight: a different subtape per shop draw

The shops are the largest exogenous swing in the game. They start empty, one unlocks when `(day + 1) % 3 == 0`, capped at eight, and the draw is with replacement, so duplicates are common and the eight slots are not eight distinct shops.

The strongest donor **carries five tapes and picks between them on whether `YARN_STORE` was drawn first, second or third**. Two others carry five consensus routes each and branch the same way.

**This is the one branch that needs no opponent model at all.** The draw is observable the turn it happens, it is the same information for both players, and it changes which products have a sink for the rest of the season. What nobody does, and it is the empty cell in the table above, is use the same signal to time selling rather than to pick a whole route.

#### Four of eight: the preempt ledger

Covered below, because it is the only edit that is structurally safe.

#### Six of eight: clone detection by public farm signature

A distance over the opponent's visible farm. The strongest donor's is

    |hands| + 3 * |unlocked quadrants| + sum over 11 tile categories

**latched after 24 consecutive near turns, starting from step 48.** Once latched it stays latched, which is the right shape: a false positive that unlatches every few turns would thrash the sell schedule.

Section 4 said the opponent's farm is fully observed from turn 0, and this reads exactly those fields. Whether it PAYS is the next subsection, and the answer surprised me.

### Cell 5, and why it is the only safe edit

The preempt ledger works like this. Sell some units earlier than the plan says, write down exactly how many you advanced, and when the plan reaches the turn those units were meant to go out, cancel exactly that quantity.

Lifetime supply is conserved to the unit. Only its timing moves.

**It is the only kind of edit that cannot break a plan's arithmetic.** A tape is a plan timed against the state it expects. Change a quantity and everything downstream, every financing chain, every purchase paid for by a sale, is running against a state that no longer matches. Change only timing, and repay it, and the accounting is untouched.

### Cell 6, and what the layer is actually for

The strongest of these agents by my own arena is one I will not name. I instrumented its layer and ran it live against three kinds of opponent.

| opponent | latches | units shifted | what happened |
|---|---:|---:|---|
| a public agent not running its tape | 0 | 0 | plays its default tape 719 of 719 turns |
| a mirror of itself | 1 | 362 | latches, shifts 362 units over 63 turns |
| a clone that preempts even earlier | 1 | 615 | escalates, shifts 615, wins 92,774 to 90,273 |

**Its adaptive machinery is anti-clone machinery.** Against anything not running its own tape it does nothing whatsoever.

That is not a flaw, it is a correct read of the field it lives in. This ladder is close to a monoculture: one clade holds 41 of the 69 submissions ever rated above 2,900, and three unrelated top teams play 67 to 80 % of one public route. If your most likely opponent is a copy of you, the highest value thing your layer can do is sell one turn before they do.

It also explains a measurement I had recorded and not understood. Scoring candidates against recorded opponents rather than live ones came out almost unbiased here, 147 of 180 games at exactly zero, and I wrote at the time that it was because these agents barely react. They react enormously. Just not to me.

**So copy what a layer does unconditionally and skip the part that waits for a twin.** An anti-clone subsystem is worth nothing to an agent nobody is cloning.

**Disabled is not refuted, and the discriminator is cheap.** Almost every one of these agents ships machinery it does not run: feed rescues implemented and switched off, a bounded market lookahead that is dead code, four complete dark subsystems in one agent alone. The easy conclusion is failed experiments. The alternative worth taking seriously is that nobody publishes their agent at full capacity, and a subsystem that is complete, precisely tuned, and sitting behind one clean flag reads less like abandonment than like a reserve. The discriminator between those stories costs one grep and one flip: turn the flag on in a copy, and either the stream moves, in which case the machinery was real and the OFF was a choice, or it does not, in which case the flag was never wired and the code is archaeology. I have found both, and the ratio is informative about the author.


### The same pattern in my own code, where the flag can be flipped

Reading somebody else's dark branch only gets you as far as the source. My own agents can be tested, and the test is worth describing because it changes what a classifier is for.

Twelve agents sat in a folder called `replay_route`. Run each against six different worlds, hash what it emits, and all twelve are identical in every world: PURE_REPLAY, twelve for twelve. The folder looked correctly named.

Six of them carried a complete decision layer behind a constant set to zero.

Run on sixteen of my own fielded builds, the census sorts them into exactly three states: five are TAPE ONLY with nothing to flip; one carries a vestigial flag that nothing ever reads, which is its own small lesson about believing a flag exists because the code mentions it; and the rest are LAYER OFF, a real switch that provably changes the stream when flipped, which is what a testable thin layer looks like from the outside.

A layer you cannot switch off is not a layer, it is the agent, and nothing about it can be attributed.


**A class measured from behaviour cannot see a switched-off layer.** A bare tape and a complete gated layer emit the same stream, so they are the same object to any instrument that only watches. Flipping the constant separates them, and it is cheap: the episode is deterministic and a game costs microseconds.

The distinction between the last two rows of that table is worth keeping separate. `v7.0_route` declares `REPAIR = 0` with a comment calling it the control that every later repair would be measured against, and **no line of the file ever reads it**. The repairs were never written. That is a declaration with no consumer, which is a different fault from a code path that does nothing, and counting reads diagnoses it without running anything.

There is also a trap in the flip test itself, and it fired on the first run. `v7.13_parent` has three switches on as shipped, `PROJECT`, `FUND` and `SELLS_FIRST`, and all three live downstream of `SELLER`. Flipping any of them alone changes nothing, because the master gate is shut. Called dead on that evidence, they would have been an experiment I believed I had run. **A switch has to be retested with every master gate held open**, and two of them then turn out to be perfectly live.

So the useful reading of a folder full of PURE_REPLAY is not that the agents decide nothing. It is that nobody has flipped anything yet.

## 5b. The ninth cell: routing between tapes

The taxonomy above assumes one tape and asks what the layer around it may touch. The strongest publicly documented agents in the competition do something the eight cells do not cover: they carry SEVERAL complete tapes and route between them on public state that the seed fixed before anyone moved.

The clean example is public: a rank-3 author published the reduced version of his agent with the mechanism explained. Ten complete recorded histories; the first shop reveal at step 72 picks a specialist, the ordered pair of the first two reveals at step 144 picks again; 23 of the 64 possible pairs carry a specialist and the rest fall back. His own evaluation prices the SECOND branch alone at +0.136 of point rate over 24,576 games on held-out seeds.

Measured from the outside, the class has a signature that separates it from every cell in the table: against different OPPONENTS at the same seed it varies on 0 of 719 turns, and across SEEDS it diverges exactly at the branch turns. It is opponent-blind and town-sighted. A divergence census keyed on opponents alone will file it as a fixed tape; one keyed on episodes will file it as adaptive; both miss that the branch variable is the town, which the seed determines and both farms see.

Two things follow. For CLASSIFIERS: test variance against opponents and against seeds separately, or the class is invisible. For BUILDERS: this is the cheapest reactivity in the game, because the trigger is public, revealed on a schedule, and never lies, and the response is a whole coherent plan rather than an edit that has to keep a foreign tape's promises.


## 6. Which part of a recording is tape and which part is layer

Everything above is read off source code, which only works for the sixteen agents whose source I hold. There are 344 recorded routes in my corpus and I will never see the code for 328 of them.

So the useful question is what can be inferred from actions alone, and the first thing you need is a way to tell tape from rim without opening either.

The experiment is the simplest one available: **hold the agent fixed, vary the opponent, and see which of its behaviours move**. Whatever is constant across five different opponents is tape. Whatever moves is layer.

Eight executable agents, each run live against five different opponents. The number is the standard deviation of that behaviour across the five.

| behaviour | sd across opponents | verdict |
|---|---|---|
| work share | 0.0003 to 0.0102 | the layer moves it, by at most 1.0 % |
| move share | 0.0001 to 0.0094 | the layer moves it, by at most 0.9 % |
| turns that sell | 0.0000 to 0.0204 | the layer moves it, by at most 2.0 % |
| **first sale** | **0.0000** | **pure tape, exactly zero in all eight** |
| sell concentration | 0.0000 to 0.0167 | the layer moves it, by at most 1.7 % |
| **land bought** | **0.0000** | **pure tape, exactly zero in all eight** |
| **labour hired** | **0.0000** | **pure tape, exactly zero in all eight** |

That table is short and it settles a lot.

**When the first sale happens, when land gets bought and how much labour gets hired are pure tape.** Identical against every opponent tried, in all eight agents, standard deviation exactly zero. The farm build and the opening of the sell schedule are decided before the game starts and nothing in the layer touches them.

**What the layer moves is at most about 2 % of the selling turns**, and it moves nothing else at all.

So the layer lives almost entirely inside the market channel and inside that channel it nudges a small minority of turns. Hold onto that for the next section, because it means anything you infer from one recording is overwhelmingly a fact about the tape, not about the agent's intelligence.

## 7. Reading an agent you cannot open

Now the 328 I have no source for.

Every turn, each unit does one of three things: work a tile, move, or nothing. Alongside that the market list carries sells and purchases. Seven numbers summarise a whole season of it: the work and move shares, how many turns sell at all, when the first sale happens, how concentrated the selling is, when land gets bought, and how much labour gets hired.

Call that the **texture**. It uses no prices, no banks, no opponent model and no source code.

Clustering by texture and then describing the clusters by texture would prove nothing, so the test is whether cluster membership predicts something the texture never saw. The obvious candidate is strength: play every route against a fixed panel and count wins.

344 recorded ladder routes, k-means on the seven texture features, then each route played both seats against 40 opponents over 6 seeds. The win rate is the column the clustering never saw.

| family | routes | win rate | sells on | concentrated | land at |
|---|---:|---:|---:|---:|---:|
| spreads the selling | 112 | **53.7 %** | 40 % | 41 % | turn 148 |
| middling | 168 | 38.8 % | 23 % | 53 % | turn 160 |
| holds, then dumps | 64 | **23.2 %** | 15 % | 80 % | turn 159 |

Thirty points of win rate between the best and worst family.

In [ ]:
# research/texture_classes.py, the three families and their centroids
FAM = [{"win_rate": 53.67, "routes": 112, "c": {"turns that sell": 0.4015, "sell concentration": 0.4135, "land bought": 0.2057}},
       {"win_rate": 38.75, "routes": 168, "c": {"turns that sell": 0.2331, "sell concentration": 0.5349, "land bought": 0.2222}},
       {"win_rate": 23.24, "routes":  64, "c": {"turns that sell": 0.1507, "sell concentration": 0.8035, "land bought": 0.2208}}]
NAMES = ["spreads the selling", "middling", "holds, then dumps"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 3.6),
                               gridspec_kw={"width_ratios": [1, 1.2]})

wr = [f["win_rate"] for f in FAM]
ax1.barh(range(len(FAM)), wr, color=RAMP[::-1], height=0.6)
ax1.set_yticks(range(len(FAM)),
               [f"{n}\n{f['routes']} routes" for n, f in zip(NAMES, FAM)])
ax1.invert_yaxis()
ax1.set_xlabel("win rate against a fixed panel, %")
ax1.set_title("Texture alone splits the field by 30 points")
ax1.grid(axis="x")
for i, v in enumerate(wr):
    ax1.text(v + 1.4, i, f"{v:.1f}%", va="center", color=INK,
             fontsize=9, fontweight="bold")
ax1.set_xlim(0, 68)

keys = ["turns that sell", "sell concentration", "land bought"]
labels = ["turns that\nsell", "sell\nconcentration", "land bought\n(turn / 720)"]
w = 0.26
for k, (nm, f) in enumerate(zip(NAMES, FAM)):
    ax2.bar([i + (k - 1) * w for i in range(3)],
            [f["c"][key] for key in keys],
            width=w - 0.02, color=RAMP[::-1][k], label=nm)
ax2.set_xticks(range(3), labels)
ax2.set_ylabel("centroid value")
ax2.set_title("And the difference is entirely in the selling")
ax2.legend(frameon=False, fontsize=8.5, loc="upper left")
ax2.grid(axis="y")
fig.tight_layout()
plt.show()

Three natural families, and they differ by **thirty points of win rate**.

The weakest holds its produce and dumps it in a burst: it sells on 15 % of turns and 80 % of the volume goes out in a tenth of the season. The strongest sells on 40 % of turns and spreads it out.

The flat group in the right hand panel does as much work as the tall ones. **Land is bought at essentially the same turn by all three families**, 148 against 159 and 160 out of 720, which is half a day apart in a thirty day season. The farm build does not separate these routes at all. The entire thirty point spread lives in the sell schedule.

That recovers, from a direction sharing nothing with it, a result I had measured the expensive way: **deferring a sale is correct when you are alone and ruinous in company**. No prices went into this, no opponent model, no source. Just what the agent did with its hands.

Two things it does not establish. These are texture families, not intentions: a route in the strongest cluster is known to look like a fast seller and to win like one, not to have been designed as one. And the panel is drawn from the same corpus, so a route is sometimes scored against its own relatives, which does not inflate anything systematically but is not the ladder either.

### What predicts whether somebody else's recording will work in your hands

Related question, different answer, and the obvious predictors all fail. One of them fails backwards.

| predictor of whether a recording transfers | r | |
|---|---:|---|
| the bank the source made | -0.028 | no relationship at all |
| the margin the source won its episode by | +0.549 | and not even monotonic |
| the source team's ladder rating | **-0.368** | NEGATIVE |


Every obvious predictor fails, and one fails backwards. The bank tells you nothing. The margin is weakly positive and not even monotonic, so ordering by it reorders wrongly in the middle where the decisions happen. And **team rating predicts transfer NEGATIVELY**: the better the team, the worse their recording works in your hands. After section 5 that stops being a paradox: strong agents are strong partly because of machinery that does not travel with the tape, and the strongest agent on this board is also the one that repeats itself least, so there is no fixed plan there to take at all.

There is a predictor that does work, it costs no games to compute, and it is not any of the three above. It is a similarity measurement over a team's own episodes, and I am holding it until the competition closes; the negative results above are the half that is safe to hand over now, and they are the half that saves you from the mistakes everyone makes first.


## 7b. The macroeconomic profile of an agent

Everything so far reads an agent's STREAM. There is a complementary reading that has been at least as useful here: treat the agent as a small economy and read its season-level aggregates, because most of the gap between a mid-ladder agent and the leader lives in quantities no single turn shows.

**The profile is a short vector, and every entry is computable from public observations**, either from downloaded replays of real games or by playing the agent against recorded rivals:

* **capital calendar**: the day each land expansion actually SETTLES (the engine refuses an underfunded purchase silently, so count owned quadrants per turn, never purchase orders);
* **labour**: hires per day, and idle-with-work unit-turns, units that PASS while watering, harvesting or planting was available;
* **production mix**: what the farm builds, herd composition included;
* **timing**: when the selling starts, how concentrated it is, how the money curve climbs by day;
* **terminal discipline**: inventory stranded at the final bell, valued at the closing book, plus tiles standing empty in the last days with seed in stock;
* **efficiency**: executed travel against a routing bound, if you want the micro end of it.

Two figures carry the idea. The first is a profile read from hundreds of real games, a mid-ladder agent against the current leader, each indicator normalised to the leader's median, one dot per real episode so the SPREAD is visible alongside the level. The instructive part is where the clouds separate and where they overlap: the mid agent buys land late with a wide spread against the leader's tight day, strands five times the inventory in clouds that barely touch, while on endgame fallow the clouds genuinely overlap even though the medians sit far apart, which a bar chart would have hidden. That is a work list, ranked, with its own error bars, and it cost no protocol runs to produce.


In [ ]:
# The same profile, shown as it actually is: a DISTRIBUTION per indicator,
# one dot per real episode (245 for the leader, 103 for the mid agent),
# jittered vertically, alpha so concentration reads as density. Each row is
# normalised to the LEADER'S MEDIAN, marked at 1; the tick above each cloud
# is that side's median. Points beyond 6x are drawn at the edge.
import json, random
DIST = json.loads('{"leader":{"land2":[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],"land3":[8,8,8,8,8,8,8,8,8,8,8,8,8,9,9,8,8,8,8,10,8,8,9,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,9,8,8,8,8,8,9,9,8,8,8,9,8,8,8,8,8,9,8,9,8,8,9,8,8,8,8,9,8,8,8,8,8,8,8,9,8,8,8,8,9,8,8,8,8,8,8,8,8,8,8,8,9,8,9,8,8,9,9,8,8,8,8,9,8,8,8,9,8,8,8,8,9,8,9,8,9,8,9,8,8,9,8,10,8,9,8,8,8,9,8,8,8,9,10,10,8,8,8,8,8,8,8,8,9,8,8,10,8,8,8,8,9,9,8,8,8,8,8,8,8,8,9,9,8,8,8,8,8,8,8,8,8,8,10,8,8,8,8,8,8,8,8,9,9,8,9,8,8,9,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,9,9,8,8,9,9,8,8,8,8,9,8,9,9,8,8,8],"pass_work":[335,339,334,326,335,336,326,336,332,330,339,346,331,340,319,331,332,322,323,324,325,329,333,337,339,342,321,332,331,317,322,320,329,325,329,324,316,320,326,328,324,322,323,320,331,321,322,316,321,323,319,325,315,328,319,326,318,314,336,316,321,321,318,321,317,325,329,319,314,317,324,331,318,320,322,324,325,328,320,316,330,317,319,313,327,325,325,323,334,318,314,327,326,322,325,317,332,317,316,324,328,313,324,316,317,328,341,337,331,325,348,341,338,328,339,337,333,331,331,328,335,328,326,333,326,336,340,330,324,341,336,329,332,332,347,326,356,336,348,338,327,331,329,326,325,333,327,337,339,331,326,329,326,331,330,343,339,320,337,324,334,335,343,328,331,330,332,335,335,351,326,332,327,321,336,332,330,333,341,344,324,328,328,340,331,337,333,328,345,332,338,338,338,330,333,331,324,347,355,341,335,348,335,348,330,341,326,328,330,336,332,330,337,332,356,328,335,330,336,333,340,339,333,325,361,339,332,329,338,333,339,330,330,348,329,336,353,340,348,340,346,328,329,331,327],"fallow_late":[9.0,30.8,15.0,9.0,11.2,16.6,9.6,17.6,10.2,10.6,22.8,12.4,16.0,21.4,0,21.4,10.6,9.6,15.2,12.2,11.4,7.8,16.2,14.2,11.2,8.4,13.0,13.2,10.6,7.4,9.8,14.2,16.2,13.0,16.6,16.2,12.0,13.0,12.8,14.2,10.2,14.0,11.2,13.0,12.6,12.0,7.8,12.0,8.5,11.0,13.6,1.0,12.4,13.0,9.0,12.8,15.8,15.2,8.6,10.8,15.2,16.6,11.4,13.4,10.4,4.5,15.0,18.4,10.8,15.2,12.0,14.4,13.0,14.4,15.4,12.2,14.0,16.6,11.6,10.6,14.4,14.6,13.0,16.2,12.0,17.2,10.6,7.6,15.0,10.0,15.6,13.4,15.6,13.2,17.6,13.4,14.4,11.2,14.0,13.4,11.8,10.8,12.4,17.0,10.4,10.4,10.4,12.6,9.8,10.8,13.4,12.8,14.8,11.4,14.6,17.2,16.0,17.2,10.0,10.6,10.2,8.0,12.8,15.8,11.2,15.0,14.0,15.4,9.0,14.4,10.6,11.8,11.0,20.0,12.8,9.0,19.0,11.2,17.4,11.8,14.4,13.4,12.8,9.6,8.8,15.2,14.4,8.5,13.8,17.2,12.0,15.4,11.6,6.0,10.2,14.6,26.6,11.8,11.8,11.0,14.8,19.6,29.8,10.8,11.0,15.6,11.4,14.6,15.2,0,12.2,17.6,12.4,10.6,14.8,1.5,13.8,16.6,15.0,15.6,6.0,15.0,9.8,21.2,16.2,10.4,9.6,10.8,7.0,11.0,10.8,13.4,10.4,9.8,13.2,13.6,8.4,22.0,9.6,18.6,12.0,13.2,13.4,8.8,11.6,16.8,7.0,14.4,16.0,15.4,16.6,13.0,13.4,13.6,14.2,11.4,14.4,13.4,13.2,17.6,14.0,15.4,14.0,7.8,6.0,18.2,14.8,8.8,17.2,11.0,22.4,11.4,6.0,22.8,11.8,18.6,15.2,16.6,13.0,13.4,20.8,10.0,16.2,12.2,11.6],"stranded":[1796,0,0,1129,132,616,1124,0,231,385,442,663,369,357,0,172,803,260,0,451,242,1276,0,723,405,624,612,238,929,316,1591,1133,192,579,650,647,900,714,1125,902,17,324,285,0,451,467,1002,271,681,40,408,0,329,183,282,637,2,276,204,112,168,0,1083,0,716,336,144,0,1375,160,330,330,1,0,256,217,1878,0,4,320,301,346,0,638,0,220,64,1107,0,1477,0,197,0,148,288,1016,0,288,35,256,905,226,270,3,598,1692,0,1188,896,1448,946,1196,973,357,1071,405,810,961,1184,1136,227,1054,1047,470,7,358,1495,0,1394,682,1008,1476,1271,0,841,0,440,1155,589,750,193,156,1174,782,1157,439,1069,0,459,467,1125,32,963,408,1026,183,0,282,1346,852,30,272,0,571,632,351,364,371,1272,0,0,249,804,1517,378,635,550,710,1080,0,411,760,526,0,843,1083,972,867,84,1316,936,1420,1302,1429,297,1780,168,0,0,168,582,870,668,666,656,1130,495,1345,1053,789,219,0,1080,495,744,418,1088,0,564,0,524,0,587,862,0,720,615,936,1,1060,0,0,210,0,507,424,870,212,0,622,410,1190,599,500,551]},"mid":{"land2":[9,12,9,9,9,9,8,9,9,9,9,9,9,8,9,9,9,9,9,9,7,9,7,9,7,6,9,7,7,7,7,6,7,7,7,7,7,7,7,7,7,7,7,6,9,7,7,7,7,9,9,9,7,7,6,9,6,6,9,9,7,9,7,7,9,7,9,9,7,7,9,9,6,7,7,9,6,7,9,9,6,7,6,9,9,9,7,9,7,7,7,9,9,9,7,9,9,9,9,6,7,6,7],"land3":[13,20,15,13,13,13,12,12,13,13,13,13,13,12,13,12,14,12,13,13,13,12,12,12,13,13,13,13,12,12,12,13,13,12,13,13,12,12,13,13,13,12,12,12,13,14,13,12,15,14,15,12,12,13,13,13,9,13,12,13,13,12,12,12,12,13,12,14,12,13,13,13,13,13,12,12,9,12,12,13,13,12,13,13,15,13,13,13,12,12,13,11,12,13,13,12,13,15,12,13,13,13,13],"pass_work":[413,392,395,390,396,392,385,381,398,390,384,383,382,417,385,408,390,375,384,393,387,400,391,404,392,398,392,404,412,401,390,388,377,403,405,393,421,412,399,388,404,408,400,381,377,378,368,420,379,385,369,395,411,380,391,391,435,397,400,388,374,387,415,388,377,405,405,401,420,387,425,381,395,384,408,415,365,421,380,389,373,403,405,385,374,394,399,388,409,413,403,414,382,409,381,401,397,406,415,376,396,391,403],"fallow_late":[29.0,27.2,31.4,22.6,28.2,24.8,26.4,27.8,30.8,29.8,27.4,34.4,24.8,22.8,23.0,31.0,28.2,25.4,23.6,26.4,31.0,32.2,30.0,30.2,24.0,27.0,27.2,29.6,34.8,31.6,28.0,26.6,28.8,34.8,26.4,28.0,36.6,33.6,28.8,24.8,29.8,28.2,34.6,36.4,21.4,27.6,29.6,34.0,33.2,28.6,25.4,26.6,36.4,25.6,26.2,23.4,29.2,27.0,32.8,20.2,22.2,29.8,29.2,26.6,26.8,25.8,28.0,29.4,33.4,19.4,27.2,28.2,21.6,24.8,37.8,23.0,32.4,27.8,25.4,30.8,18.0,34.6,23.6,22.8,29.6,25.6,26.4,26.6,35.6,31.2,26.8,30.0,23.6,26.6,23.2,30.6,26.4,23.0,27.0,27.2,32.2,24.8,30.0],"stranded":[1847,4346,2730,3942,1535,4262,2197,4412,2088,5280,4100,2992,2176,1609,5506,1584,6168,2074,2146,1151,3416,3188,2115,5199,7031,1276,1141,3511,3645,1512,4736,1316,1360,2094,4022,4826,4254,1995,4032,1650,4250,2263,1866,0,5040,3005,0,110,1877,4083,2076,2640,3657,1805,2900,2007,485,1739,3123,3858,1300,4797,2460,1545,6252,5639,1918,3818,270,2052,2189,440,5953,1549,2703,1731,1615,2212,4467,7663,6094,1286,1205,6418,5779,2510,2390,5556,3027,1211,2776,4636,5408,1368,5981,1320,2303,1104,2187,2220,2331,1206,914]}}')

rows = [('2nd quadrant settles', 'land2'),
        ('3rd quadrant settles', 'land3'),
        ('idle-with-work', 'pass_work'),
        ('fallow tiles, late days', 'fallow_late'),
        ('stranded at the bell', 'stranded')]

def median(v):
    s = sorted(v); n = len(s)
    return s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2

rng = random.Random(11)
fig, ax = plt.subplots(figsize=(9.2, 5.2))
XMAX = 6.0
for i, (label, key) in enumerate(rows):
    lm = median(DIST['leader'][key]) or 1.0
    for side, colour, dy in (('leader', '#2a78d6', +0.16), ('mid', '#eb6834', -0.16)):
        vals = [min(v / lm, XMAX) for v in DIST[side][key]]
        ys = [i + dy + rng.uniform(-0.11, 0.11) for _ in vals]
        ax.scatter(vals, ys, s=9, color=colour, alpha=0.18, linewidths=0)
        m = min(median(DIST[side][key]) / lm, XMAX)
        ax.plot([m, m], [i + dy - 0.14, i + dy + 0.14], color=colour, lw=2.4)
ax.axvline(1.0, color='#222', lw=1.1)
ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows], fontsize=9)
ax.invert_yaxis()
ax.set_xlim(-0.15, XMAX + 0.25)
ax.set_xlabel('multiple of the leader median for that indicator (1 = the leader median)')
ax.set_title('A macro profile is a distribution, never a point')
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], marker='o', ls='', color='#2a78d6', label='leader, per episode'),
                   Line2D([], [], marker='o', ls='', color='#eb6834', label='mid-ladder agent, per episode')],
          frameon=False, loc='lower right', fontsize=8.5)
plt.tight_layout(); plt.show()
print('None of these is a constant, and the clouds correct two readings the')
print('bars invited. Idleness is a SMALL but consistent gap: both clouds are')
print('tight and clearly apart at about 1.2x, no overlap to hide behind.')
print('Fallow is the opposite: the medians sit far apart but the clouds')
print('genuinely overlap between 1.5x and 2.4x, so single episodes prove')
print('nothing there. And stranded inventory is a long one-sided tail, the')
print('mid agent reaching past six times the leader median while the leader')
print('cloud hugs zero: a structural gap, carried by the tail.')


The second figure is the DYNAMIC version, and it is the single most diagnostic chart this project produces. Take one tape, play it twice on the same world, once alone and once against a rival that sells into the same book, and plot its money at each day's end. The two runs are identical software; the gap that opens is pure economics, and WHERE it opens tells you the mechanism: here the curves separate in the first week, while the farm is being financed, and then merely accumulate the damage. The season was lost before day ten in refusals of a few coins each.


In [ ]:
# Day-end money of the SAME tape on the SAME world: alone, and against a
# co-selling rival. Real forensic data from one replayed pair of games.
alone   = [24, 93, 209, 282, 233, 40, 744, 285, 1410, 2960, 18497, 18437,
           24195, 26980, 31095, 35309, 42840, 46931, 56821, 62497, 71864,
           80355, 92780, 100301, 106517, 115365, 122308, 128900, 137228]
company = [19, 105, 219, 267, 190, 0, 264, 241, 0, 213, 8151, 4890, 5745,
           7110, 9708, 11498, 12953, 15481, 17790, 18395, 20528, 23401,
           24362, 26271, 27647, 29841, 31895, 33319, 37096]

fig, ax = plt.subplots(figsize=(8.6, 3.8))
d = range(len(alone))
ax.plot(d, alone, color=BLUE, lw=2, label='alone on the world')
ax.plot(d, company, color=ORANGE, lw=2, label='same tape, co-selling rival')
ax.fill_between(d, company, alone, color=GRID, alpha=0.6)
ax.set_xlabel('day'); ax.set_ylabel('money at day end')
ax.set_title('the gap opens during financing, then only accumulates', loc='left')
ax.legend(frameon=False, loc='upper left')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()
print('days 5-9 in company: cash pinned at zero, purchases silently refused,')
print('and by day 10 the farm is half-built. The bank at the bell differs by')
print('107,000; the decision that lost it cost about forty coins at a time.')


And under the profile sit the DERIVED quantities, the layer between the engine's constants and anybody's choices. They are worth keeping as an explicit list, because each one converts a tuning question into a reading question: the per-product drain the town exerts, which is a rule you can transcribe rather than a rate you estimate; the sale lot that a price curve tolerates, which is the curve's slope at the book rather than a constant; the true cost of feed for a farm that grows its own, which is seed plus labour over yield rather than the market quote; the near-free price of labour at small crews against its convex climb; and the season sink per product, which decides which lines can absorb volume at all. An agent's profile is largely a record of which of these its author derived and which they guessed.


## 8. Where the money actually is

The same tape, valued two ways: playing alone against an empty market, and playing in company against a real opponent.

| milk | alone | in company | ratio |
|---|---:|---:|---:|
| \$ per unit | 271.0 | **37.6** | **0.14** |

Alone, the price climbs all season, 169 on day 1 to 290 on day 30, so deferring pays. In company it collapses to two to five per cent of that from day 18 on, so deferring is ruin. This tape sells 196 of its 328 milk units from day 18 onward.

Deferring is optimal alone and catastrophic in company, and every recorded tape defers, because every recorded tape was optimised against something.

The consequence for anyone building here: **an instrument pointed at an idle market is measuring a different game.** An agent can bank 112,000 against nothing and 28,000 against a real opponent on the same seed, and look healthy in every test you ran.

I spent three separate attempts building decision layers at three different time scales, cycle, preempt and advance, and all three failed the same bar. Then a production census showed why: the tape already sells milk the moment it exists. Timing was never the binding constraint there. What separates two tapes is **which goods they supply into a shared book**, and that is composition, fixed on day zero by the farm build, which is cell 7 of section 5 and the one cell no layer is licensed to touch.

**The existence proof needs no statistics.** One tape banks LESS in company than a tape it beats, 77,392 against 86,606, and the whole difference is what each one allowed the rival to bank alongside it, 66,755 against 96,054. Neither tape is worse at farming. One of them leaves more on the table for whoever sits opposite, and the game pays for the difference between the two banks, not for the size of yours. Every aggregate in this section is a statistical restatement of that one pair of games.


### Which is why picking the recording dominates everything else

Four candidate founders scored on the population that actually decides a ladder placement, rather than on a panel of strong opponents.

344 recorded ladder routes, 12 seeds, both seats, 8,256 games each. Intervals clustered on the ROUTE rather than on the game.

| founder | win rate | 95 % clustered | if treated as independent |
|---|---:|---|---|
| founder A | **86.0 %** | [84.2, 87.7] | [85.2, 86.7] |
| founder B | 74.5 % | [73.0, 75.9] | [73.5, 75.4] |
| founder C | 39.0 % | [35.3, 42.8] | [38.0, 40.1] |
| founder D | **12.8 %** | [10.2, 15.4] | [12.1, 13.6] |

In [ ]:
# research/population_control.py: name, win rate, clustered low, clustered high
POP = [("founder", 85.96, 84.21, 87.72),
       ("founder",  74.45, 72.98, 75.93),
       ("founder",    39.05, 35.33, 42.77),
       ("founder",       12.81, 10.18, 15.45)]

fig, ax = plt.subplots(figsize=(7.6, 3.1))
names = [p[0] for p in POP][::-1]
wr = [p[1] for p in POP][::-1]
lo = [p[1] - p[2] for p in POP][::-1]
hi = [p[3] - p[1] for p in POP][::-1]

upper = [p[3] for p in POP][::-1]                # label past the whisker, not the bar
ax.barh(names, wr, color=BLUE, height=0.55)
ax.errorbar(wr, names, xerr=[lo, hi], fmt="none", ecolor=INK, capsize=4, lw=1.4)
for n, v, u in zip(names, wr, upper):
    ax.text(u + 2.5, n, f"{v:.1f}%", va="center", color=INK,
            fontsize=9, fontweight="bold")
ax.set_xlim(0, 100)
ax.set_xlabel("win rate over 344 recorded ladder routes, 12 seeds, both seats")
ax.set_title("Choosing which season to record spans 73 points")
ax.grid(axis="x")
fig.tight_layout()
plt.show()

Seventy three points, from the same engine and the same layer, differing only in which recording is underneath.

Two cautions on that figure.

**Those 8,256 games are not 8,256 independent trials.** Each of the 344 routes contributes 24 of them, so treating them as independent gives an interval between two and four times too narrow, which is the gap between the two columns in the table. The bars are clustered on the route.

**And the population decides the answer.** I originally ranked these same four founders against a panel of the five strongest public agents, because scoring against the hardest available opponents felt like rigour. The ladder does not pay for beating the five best agents in the competition, it pays for beating whoever it matches you against, and those are ordinary submissions. Re-scored on the corpus above the ordering does not soften, **it inverts**: the founder the panel put first came third, and the one it put near last came first by eleven points.

## 9. The ceiling of the layer, measured from above

Sections 5 to 8 say the layer lives in the market channel and moves a small minority of turns. So how much could a *perfect* market layer be worth? There is a clean way to ask.

Freeze one tape and one recorded opponent so the episode is deterministic. Vary only my own market list, **with full hindsight of what the rival actually did**, and hill climb. That is information no agent can have, so whatever it finds is an upper bound on what any policy of that shape could win.

Split three ways, because the decomposition is the point and the single number is not: permute the order within a turn, change the quantities, or move an order to a different turn.

28 recorded opponents, 12 seeds, 400 hill-climb steps per arm per game, 336 games each. A typical bank in this game is of order \$85,000.

| arm | mean | median | p90 |
|---|---:|---:|---:|
| permute the order within the turn | 2,032 | 68 | 3,387 |
| change how much each order sells | 1,398 | 142 | 2,823 |
| **move an order to another turn** | 4,006 | 1,419 | 10,127 |


In [ ]:
# research/market_oracle.py: mean, median, p90 and n per arm
ARMS = {"index": [2032.4, 68.0, 3387.0, 336], "quantity": [1398.5, 142.0, 2823.0, 336], "turn": [4006.5, 1419.0, 10127.0, 336]}
LAB = {"index": "permute order within the turn",
       "quantity": "change how much each order sells",
       "turn": "move an order to another turn"}

fig, ax = plt.subplots(figsize=(7.6, 2.9))
ks = ["index", "quantity", "turn"]
best = max(ks, key=lambda k: ARMS[k][0])
ax.barh([LAB[k] for k in ks], [ARMS[k][0] for k in ks],
        color=[ORANGE if k == best else BLUE for k in ks], height=0.55)
for y, k in enumerate(ks):
    ax.text(ARMS[k][0] + ARMS[best][0] * 0.04, y,
            "\\$" + f"{ARMS[k][0]:,.0f} mean, "
            + "\\$" + f"{ARMS[k][1]:,.0f} median",
            va="center", color=INK, fontsize=8.5)
ax.set_xlim(0, ARMS[best][0] * 2.0)
ax.set_xlabel(r"\$ gained per game WITH hindsight, so an upper bound")
ax.set_title(f"Even clairvoyant, the best arm is worth "
             f"{ARMS[best][0] / 85_000:.0%} of a bank")
ax.grid(axis="x")
fig.tight_layout()
plt.show()

**Every arm is small, and two of them are close to nothing.** Against banks of order 85,000, the median game gains 0.08 % from reordering and 0.17 % from requantising. The best arm reaches about 1.7 % in the median and 4.7 % on the mean, so the mean is being carried by a minority of games rather than describing a typical one. Set any of that beside the 73 point founder spread in section 6 and the ordering of work is settled: the market list is not where this game is decided.

It sits next to a number that looks like it says the opposite. Deleting one order from a recorded tape is worth about 95,000 dollars a game. But that is a *catastrophic failure* sensitivity, not an optimisation headroom: it measures what happens when a financing chain breaks and half the farm stays locked for the season. Once the chain is intact, rearranging it is worth almost nothing. Both facts are true and I had been quoting the first as though it implied the second.

**The arm that carries the most is the timing arm.** Three timing layers had failed my bar, and I had read that as timing not being the binding constraint. The ceiling on timing is the largest of the three, so what those layers failed against was more likely the evaluation than the game. A measurement built only to refute has reopened a conclusion instead of confirming one.

The timing arm can also change the order of orders at the turn it moves them into, so it is not perfectly disjoint from the index arm. It is an upper bound on timing plus a little index, and the honest statement is that timing dominates within that overlap rather than cleanly beside it.

## 9b. Two ways to stand on a recording, and why an owned one is a moat

Everything in this guide supports two distinct construction strategies, and they are worth stating as strategies rather than leaving implicit.

**The first: a top public recording as the base, your thin layer on it.** The fastest possible route to a competitive agent; the tape arrives pre-validated by the ladder itself, and your layer is confined to the classes section 5 licenses. Its ceiling is also its nature: the base is PUBLIC. Whoever else stands on the same recording plays mirror games against you, ties at the margin, and the strongest layers in the field carry machinery aimed precisely at rivals running a tape they can recognise. You inherit the strength of the commons and its exposure at the same time.

**The second: an OWNED schedule as the base, the same thin layer on top.** Take a recording, or the space of schedules a recording implies, and SEARCH it: mutate the plan under an evolutionary loop, score populations of variants against pools of real rivals, keep what survives out of sample. That is expensive in exactly the place the first strategy is free, and it buys three things the first strategy cannot have.

It buys privacy: nobody can pull your schedule from a public notebook, so the mirror and anti-clone machinery that patrols this monoculture has nothing to latch onto. It buys fit: the search optimises against the field as it stands, not the field a public tape was recorded in. And it buys compounding: every micro-improvement the search finds stays yours, where an edit to a public tape is one download away from everyone.

That is the sense in which **an owned schedule is a moat**. The public-tape strategy competes on the LAYER, and section 9 bounds what a layer can ever be worth; the owned-schedule strategy competes on the BASE, which section 8 shows is where seventy-plus points of win rate live. The moat has a maintenance cost, the field's half life is measured in days and an owned schedule ages like any other, but what it loses to time it does not lose to copying.

The honest footnote from having run both: the search's own instrument is the hard part. A schedule selected against too few seeds or the wrong population arrives pre-overfitted, and every lesson in sections 7 and 8 applies to the search loop with the volume turned up.


## 10. What I would tell you, depending on where you are

**If you are starting.** A recording plus twenty lines is already a competitive agent, and the clock bug in section 1 is the one that will get you: derive the turn from `day * 24 + hour`. Then read section 2 twice, because the market list resolving by index is the fact that makes tapes work at all and the one that punishes you for editing them casually.

**If you are building a layer.** Look at the eight cells and notice which one you are in, and check the ninth first: if your trigger is something the seed fixed and both farms can see, you are not editing a tape, you are choosing between tapes, and that class holds the top of this ladder. If you are in cell 6 building opponent response, check first whether your opponent is a clone of you, because that machinery is dormant against everybody else. Cell 3 is empty and cheap. Cell 1 is unglamorous and is most of the value.

**If you are measuring.** Check what your candidate does against a live opponent rather than an empty market, because the gap between those numbers is larger than anything a layer will recover, and the mechanism behind most of it is the financing cascade in section 4: refusals of forty dollars, not battles. And write down which population you scored against, because ranking the same four candidates against a panel of the strongest agents rather than against the ladder does not soften the ordering, it inverts it.

**And the general one.** Almost everything in here that turned out to be true was measurable before writing any code, and almost everything that wasted time was built first and evaluated second. The engine is deterministic and a game costs microseconds in batch. There is very little excuse for guessing.